In [15]:
# ============================================================
# SETUP BATCH EVENTS — LOCAL EXECUTION
# ============================================================

import os
from datetime import datetime
from pyspark.sql import Row
from pyspark.sql import functions as F

In [16]:
# -----------------------------------------
# 1. IMPORT ENVIRONMENT CONFIG IF NOT LOADED
# -----------------------------------------
try:
    CONTROL_ROOT
except NameError:
    import importlib.util
    import nbformat
    from nbconvert import PythonExporter

    config_path = "../00-common/01_environment_config.ipynb"
    with open(config_path, "r") as f:
        nb = nbformat.read(f, as_version=4)

    exporter = PythonExporter()
    source, _ = exporter.from_notebook_node(nb)
    exec(source)

In [17]:
# -----------------------------------------
# 2. Define batch events file path
# -----------------------------------------
batch_events_path = f"{CONTROL_ROOT}/batch_events.parquet"

In [18]:
# -----------------------------------------
# 3. If file does not exist → create first batch
# -----------------------------------------
if not os.path.exists(batch_events_path):

    # Use Python timestamp, NOT F.current_timestamp()
    first_batch = spark.createDataFrame(
        [Row(batch_id=1, event_timestamp=datetime.now())]
    )

    first_batch.write.mode("overwrite").parquet(batch_events_path)
    print("✔ Created batch_events.parquet with batch_id = 1")

# -----------------------------------------
# 4. If exists → append new batch
# -----------------------------------------
else:
    existing = spark.read.parquet(batch_events_path)
    max_id = existing.agg(F.max("batch_id")).first()[0]

    # Use Python timestamp again
    new_batch = spark.createDataFrame(
        [Row(batch_id=max_id + 1, event_timestamp=datetime.now())]
    )

    new_batch.write.mode("append").parquet(batch_events_path)
    print(f"✔ Appended new batch_id = {max_id + 1}")

✔ Appended new batch_id = 2


In [19]:
# -----------------------------------------
# 5. Show table
# -----------------------------------------
spark.read.parquet(batch_events_path).show()

+--------+--------------------+
|batch_id|     event_timestamp|
+--------+--------------------+
|       2|2026-06-08 21:12:...|
|       1|2026-06-08 21:11:...|
+--------+--------------------+

